In [1]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc

a = 1.20577 # bond length in a cluster
d = 4 # distance between each cluster
unit = 'A' # unit of length
na = 2 # size of a cluster (monomer)
nc = 2 # set as integer multiple of monomers
spin = 2 # spin per monomer
elmt = 'O'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"
###########################

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=20000,
            )
mol.build()

# mol = gto.Mole()
# mol.verbose = 4
# mol.atom = '''
# O   -1.485163346097   -0.114724564047    0.000000000000
# H   -1.868415346097    0.762298435953    0.000000000000
# H   -0.533833346097    0.040507435953    0.000000000000
# O    1.416468653903    0.111264435953    0.000000000000
# H    1.746241653903   -0.373945564047   -0.758561000000
# H    1.746241653903   -0.373945564047    0.758561000000
# '''
# mol.basis = 'cc-pvdz'
# mol.precision = 1e-10
# mol.build()

mf = scf.UHF(mol).density_fit()
mf.kernel()

# frozen = 0
mymp = mp.MP2(mf).set_frozen()
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-28-generic', version='#28~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Wed Jul  1 15:50:57 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Thu Jul 30 13:56:13 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 4
[INPUT] num. electrons = 32
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 4
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = A
[INPUT] Symbol           X                Y                Z      unit

In [2]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}') 

MP2 Corr = -0.19432810
CCSD Corr = -0.21781221
CCSD(T) Corr = -0.21921819


In [3]:
import jax
jax.config.update("jax_enable_x64", True)

from pyscf import lib
from pyscf.lno import lnoccsd, ulnoccsd
from jax import random
from afqmc.lno_afqmc import prep

In [4]:
from afqmc.lno_afqmc import lno_afqmc, tools
from pyscf.data import elements
iao_coeff, iao_frag_list, atm_center = tools.iao_localization(mf)

In [5]:
options = {
           'eql_time': 20,
           'n_prop_steps': 50,
           'n_blocks': 600,
           'n_walkers': 300,
           'mix_precision': False,
           'seed': 17,
           'walker_type': 'rhf',
           'trial': 'pt2ccsd',
           }

lo_coeff = iao_coeff
frag_lolist = iao_frag_list
nfrozen = elements.chemcore(mol)
thresh = 1e-5
qmc_options = options
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = None
atom_group = atm_center
plot_las = False

    
print("\n ******* LNO-CALCULATION ******* \n")

tools.check_span(mf, lo_coeff, nfrozen, thresh=1e-10)

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]


mlno = lno_afqmc.get_lnoccsd(mf, lo_coeff, frag_lolist, nfrozen, thresh, spin_type)
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)

lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag_tot

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_tot,), 
                        minval=0, 
                        maxval=100*nfrag_tot
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
lno_emp = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol

# Loop over fragment
for ifrag, frag_idx in enumerate(run_frag_list):
    
    loidx = frag_lolist[ifrag]

    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {frag_idx+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[frag_idx]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    orbloc, lno_param = lno_afqmc.get_lnoparam(lo_coeff, lno_thresh, lno_pct_occ, lno_norb, loidx, ifrag, spin_type)
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

    maskact, lno_active, nactocc, nactvir, lno_tot = \
        lno_afqmc.get_las(mlno, orbloc, uocc_loc, lno_frozen, spin_type, loc_ctr)
            
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx=frag_idx+1)

    
    eorb_cc, t1, t2 = \
        lno_afqmc.lnoccsd_kernel(mlno, lno_coeff, lno_frozen, uocc_loc, maskact, verbose=4)
    eorb_mp = lno_afqmc.lnomp2_kernel(mlno, lno_coeff, lno_frozen, uocc_loc, maskact, verbose=4)

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    lno_emp[ifrag] = eorb_mp
    lno_ecc[ifrag] = eorb_cc


 ******* LNO-CALCULATION ******* 

LO occ span the occupied MO occ space - True.
MO occ span the occupied LO occ space - False.


====================== unrestricted LNO-FRAGMENT 1/(4,4) =======================
Center Atom O
PySCF NumPy Threads = 16
LAS occupied orbitals:  [7, 5]
LAS virtual orbitals:   [1, 3]
LAS total size:         [8, 8]

WARN: CCSD detected DF being used in the HF object. MO integrals are computed based on the DF 3-index tensors.
It's recommended to use dfccsd.CCSD for the DF-CCSD calculations


******** <class 'pyscf.lno.ulnoccsd.MODIFIED_UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(7), np.int64(5)), nmo = (8, 8)
frozen orbitals [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 19]), array([ 0,  1,  2,  3,  4,  5,  6,  7,  8, 17, 18, 19])]
max_cycle = 50
direct = 0
conv_tol = 1e-06
conv_tol_normt = 1e-05
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 20000 MB (current use 928 MB)
Init t2, MP2 energy = -0.0970707265676581
Init E_

In [6]:
print(lno_emp)
print(lno_ecc)

[-0.04857551 -0.04853125 -0.04853125 -0.04857551]
[-0.05446255 -0.05441241 -0.05441241 -0.05446255]


In [12]:
print(lno_emp)
print(lno_ecc)

[-0.04857551 -0.04853125 -0.04853125 -0.04857551]
[-0.05446248 -0.05441234 -0.05441234 -0.05446248]


In [8]:
from pyscf.lno import lno, ulno
from pyscf.lib import logger
from functools import reduce

def make_rlas(mlno, eris, orbloc, lno_type, lno_thresh=None):
    log = logger.new_logger(mlno)
    cput1 = (logger.process_clock(), logger.perf_counter())

    s1e = mlno.s1e

    if lno_thresh is None:
        thresh_occ, thresh_vir = mlno.lno_thresh
    else:
        thresh_occ, thresh_vir = lno_thresh

    s1e = mlno.s1e

    orboccfrz_core, orbocc, orbvir, orbvirfrz_core = mlno.split_mo_coeff()
    moeocc, moevir = mlno.split_mo_energy()[1:3]

    ''' Projection of LO onto occ and vir
    '''
    uocc_loc = reduce(np.dot, (orbloc.T.conj(), s1e, orbocc)) # <loc|mo_occ>
    # uocc_loc[act], std, frz
    uocc_loc, uocc_std, uocc_orth = \
            lno.projection_construction(uocc_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
    if uocc_loc.shape[1] == 0:
        log.error('LOs do not overlap with occupied space. This could be caused '
                  'by either a bad fragment choice or too high of `lo_proj_thresh_active` '
                  '(current value: %s).', mlno.lo_proj_thresh_active)
        raise RuntimeError
    log.info('LO occ proj: %d active | %d standby | %d orthogonal',
             *[u.shape[1] for u in [uocc_loc,uocc_std,uocc_orth]])

    uvir_loc = reduce(np.dot, (orbloc.T.conj(), s1e, orbvir))
    uvir_loc, uvir_std, uvir_orth = \
            lno.projection_construction(uvir_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
    log.info('LO vir proj: %d active | %d standby | %d orthogonal',
             *[u.shape[1] for u in [uvir_loc,uvir_std,uvir_orth]])
    if uvir_loc.shape[1] == 0:
        uvir_loc = uvir_std = uvir_orth = None

    ''' LNO construction
    '''
    dmoo = mlno.make_lo_rdm1_occ(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[0])
    if mlno._match_oldcode: dmoo *= 0.5 # TO MATCH OLD LNO CODE
    dmoo = reduce(np.dot, (uocc_orth.T.conj(), dmoo, uocc_orth))
    if lno_param[0]['norb'] is not None:
        lno_param[0]['norb'] -= uocc_loc.shape[1] + uocc_std.shape[1]
    uoccact_orth, uoccfrz_orth = lno.natorb_select(dmoo, uocc_orth, thresh=thresh_occ)
    uoccact_orth = uoccact_orth[:,::-1] # for occ. flip the NOs so they are in the
    uoccfrz_orth = uoccfrz_orth[:,::-1] # order of small eigenvalue -> large eigenvalue
    orboccfrz = np.hstack((orboccfrz_core, np.dot(orbocc, uoccfrz_orth)))
    uoccact = np.hstack((uoccact_orth, uocc_std, uocc_loc))
    orboccact = np.dot(orbocc, uoccact)
    uoccact_loc = np.linalg.multi_dot((orboccact.T.conj(), s1e, orbloc))
    can_uoccact = lno.subspace_eigh(np.diag(moeocc), np.hstack((uoccact_orth, uocc_std, uocc_loc)))[1]
    can_orboccact = np.dot(orbocc, can_uoccact)
    can_uoccact_loc = np.linalg.multi_dot((can_orboccact.T.conj(), s1e, orbloc))
    cput1 = log.timer_debug1('make_lo_rdm1_occ', *cput1)

    dmvv = mlno.make_lo_rdm1_vir(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[1])
    if mlno._match_oldcode: dmvv *= 0.5 # TO MATCH OLD LNO CODE
    if uvir_orth is not None:
        dmvv = reduce(np.dot, (uvir_orth.T.conj(), dmvv, uvir_orth))
        if lno_param[1]['norb'] is not None:
            lno_param[1]['norb'] -= uvir_loc.shape[1] + uvir_std.shape[1]
        uviract_orth, uvirfrz_orth = lno.natorb_select(dmvv, uvir_orth, thresh=thresh_vir)
        orbvirfrz = np.hstack((np.dot(orbvir, uvirfrz_orth), orbvirfrz_core))
        uviract = np.hstack((uvir_loc, uvir_std, uviract_orth)) # in decaying activity order
        orbviract = np.dot(orbvir, uviract)
        can_uviract = lno.subspace_eigh(np.diag(moevir), np.hstack((uvir_loc, uvir_std, uviract_orth)))[1]
        can_orbviract = np.dot(orbvir, can_uviract)
    else:
        orbviract, orbvirfrz = lno.natorb_select(dmvv, orbvir, thresh=thresh_vir)
        orbvirfrz = np.hstack((orbvirfrz, orbvirfrz_core))
        uviract = reduce(np.dot, (orbvir.T.conj(), s1e, orbviract))
        orbviract = np.dot(orbvir, uviract)
        can_uviract = lno.subspace_eigh(np.diag(moevir), uviract)[1]
        can_orbviract = np.dot(orbvir, can_uviract)
    cput1 = log.timer_debug1('make_lo_rdm1_vir', *cput1)

    ''' LAS construction
    '''
    orbfragall = [orboccfrz, orboccact, orbviract, orbvirfrz]
    can_orbfragall = [orboccfrz, can_orboccact, can_orbviract, orbvirfrz]
    orbfrag = np.hstack(orbfragall)
    can_orbfrag = np.hstack(can_orbfragall)
    norbfragall = np.asarray([x.shape[1] for x in orbfragall])
    locfragall = np.cumsum([0] + norbfragall.tolist()).astype(int)
    frzfrag = np.concatenate((
        np.arange(locfragall[0], locfragall[1]),
        np.arange(locfragall[3], locfragall[4]))).astype(int)
    frag_msg = '%d/%d Occ | %d/%d Vir | %d/%d MOs' % (
                    norbfragall[1], sum(norbfragall[:2]),
                    norbfragall[2], sum(norbfragall[2:4]),
                    sum(norbfragall[1:3]), sum(norbfragall)
                )
    if len(frzfrag) == 0:
        frzfrag = 0

    return orbfrag, can_orbfrag, frzfrag, uoccact_loc, can_uoccact_loc, frag_msg


def make_ulas(mlno, eris, orbloc, lno_type, lno_thresh=None):
    """
    Create localized active space for a given set of localized orbitals
    given in orbloc
    """
    log = logger.new_logger(mlno)

    if lno_thresh is None:
        thresh_occ, thresh_vir = mlno.lno_thresh
    else:
        thresh_occ, thresh_vir = lno_thresh

    s1e = mlno.s1e

    orboccfrz_core = [None,] * 2
    orbocc = [None,] * 2
    orbvir = [None,] * 2
    orbvirfrz_core = [None,] * 2
    moeocc = [None,] * 2
    moevir = [None,] * 2

    uocc_loc = [None,] * 2
    uocc_std = [None,] * 2
    uocc_orth = [None,] * 2

    mo_splits = mlno.split_mo_coeff()
    moe_splits = mlno.split_mo_energy()
    for s in range(2):
        orboccfrz_core[s], orbocc[s], orbvir[s], orbvirfrz_core[s] = mo_splits[s]
        moeocc[s], moevir[s] = moe_splits[s][1:3]
        
        #####################################
        # Projection of LO onto occ and vir #
        #####################################
        ovlp = reduce(np.dot, (orbloc[s].T.conj(), s1e, orbocc[s]))
        uocc_loc[s], uocc_std[s], uocc_orth[s] = \
            lno.projection_construction(ovlp, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
        # NOTE we allow empty fragments
        # if uocc_loc[s].shape[1] == 0:
        #    log.error('LOs do not overlap with occupied space. This could be caused '
        #              'by either a bad fragment choice or too high of `lo_proj_thresh_active` '
        #              '(current value: %s).', mlno.lo_proj_thresh_active)
        #    raise RuntimeError
        log.info('LO occ proj: %d active | %d standby | %d orthogonal',
                 *[u.shape[1] for u in [uocc_loc[s], uocc_std[s], uocc_orth[s]]])

    ####################
    # LNO construction #
    ####################
    if lno_type[0] == lno_type[1] == '1h':
        # NOTE: uvir_loc is not used in 1h/1h, so we pass None
        if getattr(mlno, 'with_df', None):
            dmoo, dmvv = ulno.make_lo_rdm1_1h_df(eris, moeocc, moevir, uocc_loc)
        else:
            dmoo, dmvv = ulno.make_lo_rdm1_1h(eris, moeocc, moevir, uocc_loc)
    else:
        raise NotImplementedError('Unsupported LNO type')
        
    # if mlno._match_oldulno:
    #     dmoo[0],dmoo[1]=dmoo[0]/2.0,dmoo[1]/2.0
    #     dmvv[0],dmvv[1]=dmvv[0]/2.0,dmvv[1]/2.0

    orbfrag = [None,] * 2
    frzfrag = [None,] * 2
    uoccact_loc = [None,] * 2
    can_orbfrag = [None,] * 2
    can_uoccact_loc = [None,] * 2
    frag_msg = ""

    for s in range(2):
        dmoo[s] = reduce(np.dot, (uocc_orth[s].T.conj(), dmoo[s], uocc_orth[s]))

        _param = lno_param[s][0]
        if _param['norb'] is not None:
            _param['norb'] -= uocc_loc[s].shape[1] + uocc_std[s].shape[1]

        uoccact_orth, uoccfrz_orth = lno.natorb_select(dmoo[s], uocc_orth[s], thresh=thresh_occ)
        uoccact_orth = uoccact_orth[:,::-1] # for occ. flip the NOs so they are in the
        uoccfrz_orth = uoccfrz_orth[:,::-1] # order of small eigenvalue -> large eigenvalue
        orboccfrz = np.hstack((orboccfrz_core[s], np.dot(orbocc[s], uoccfrz_orth)))
        uoccact = np.hstack((uoccact_orth, uocc_std[s], uocc_loc[s]))
        orboccact = np.dot(orbocc[s], uoccact)
        uoccact_loc[s] = np.linalg.multi_dot((orboccact.T.conj(), s1e, orbloc[s]))
        # canonized
        can_uoccact = lno.subspace_eigh(np.diag(moeocc[s]), np.hstack((uoccact_orth, uocc_std[s], uocc_loc[s])))[1]
        can_orboccact = np.dot(orbocc[s], can_uoccact)
        can_uoccact_loc[s] = np.linalg.multi_dot((can_orboccact.T.conj(), s1e, orbloc[s]))

        orbviract, orbvirfrz = lno.natorb_select(dmvv[s], orbvir[s], thresh=thresh_vir)
        orbvirfrz = np.hstack((orbvirfrz, orbvirfrz_core[s]))
        uviract = reduce(np.dot, (orbvir[s].T.conj(), s1e, orbviract))
        uviract = uviract
        orbviract = np.dot(orbvir[s], uviract)
        # canonized
        can_uviract = lno.subspace_eigh(np.diag(moevir[s]), uviract)[1]
        can_orbviract = np.dot(orbvir[s], can_uviract)

        ####################
        # LAS construction #
        ####################
        orbfragall = [orboccfrz, orboccact, orbviract, orbvirfrz]
        can_orbfragall = [orboccfrz, can_orboccact, can_orbviract, orbvirfrz]
        orbfrag[s] = np.hstack(orbfragall)
        can_orbfrag[s] = np.hstack(can_orbfragall)
        norbfragall = np.asarray([x.shape[1] for x in orbfragall])
        locfragall = np.cumsum([0] + norbfragall.tolist()).astype(int)
        frzfrag[s] = np.concatenate((
            np.arange(locfragall[0], locfragall[1]),
            np.arange(locfragall[3], locfragall[4]))).astype(int)
        frag_msg += '\nSpin channel %d: %d/%d Occ | %d/%d Vir | %d/%d MOs\n' % (
                        s,
                        norbfragall[1], sum(norbfragall[:2]),
                        norbfragall[2], sum(norbfragall[2:4]),
                        sum(norbfragall[1:3]), sum(norbfragall)
                    )
        if len(frzfrag[s]) == 0:
            frzfrag[s] = 0

    return orbfrag, can_orbfrag, frzfrag, uoccact_loc, can_uoccact_loc, frag_msg

def make_las(mlno, eris, orbloc, lno_type, lno_thresh=None):
    if isinstance(mlno._scf, scf.rhf.RHF):
        return make_rlas(mlno, eris, orbloc, lno_type, lno_thresh)
    elif isinstance(mlno._scf, scf.uhf.UHF):
        return make_ulas(mlno, eris, orbloc, lno_type, lno_thresh)

In [10]:
options = {
           'eql_time': 20,
           'n_prop_steps': 50,
           'n_blocks': 600,
           'n_walkers': 300,
           'mix_precision': False,
           'seed': 17,
           'walker_type': 'rhf',
           'trial': 'pt2ccsd',
           }

lo_coeff = iao_coeff
frag_lolist = iao_frag_list
nfrozen = elements.chemcore(mol)
thresh = 1e-5
qmc_options = options
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = None
atom_group = atm_center
plot_las = False

    
print("\n ******* LNO-CALCULATION ******* \n")

tools.check_span(mf, lo_coeff, nfrozen, thresh=1e-10)

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]


mlno = lno_afqmc.get_lnoccsd(mf, lo_coeff, frag_lolist, nfrozen, thresh, spin_type)
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)

lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag_tot

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_tot,), 
                        minval=0, 
                        maxval=100*nfrag_tot
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
lno_emp = np.zeros(nfrag_run, dtype='float64')
lno_emp2 = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol

# Loop over fragment
for ifrag, frag_idx in enumerate(run_frag_list):
    
    loidx = frag_lolist[ifrag]

    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {frag_idx+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[frag_idx]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    orbloc, lno_param = lno_afqmc.get_lnoparam(lo_coeff, lno_thresh, lno_pct_occ, lno_norb, loidx, ifrag, spin_type)
    lno_coeff, can_lno_coeff, lno_frozen, uocc_loc, can_uocc_loc, _ \
        = make_las(mlno, eris, orbloc, lno_type)

    maskact, lno_active, nactocc, nactvir, lno_tot = \
        lno_afqmc.get_las(mlno, orbloc, uocc_loc, lno_frozen, spin_type, loc_ctr)
            
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx=frag_idx+1)

    
    eorb_cc, t1, t2 = \
        lno_afqmc.lnoccsd_kernel(mlno, lno_coeff, lno_frozen, uocc_loc, maskact, verbose=4)
    eorb_mp = lno_afqmc.lnomp2_kernel(mlno, can_lno_coeff, lno_frozen, can_uocc_loc, maskact, verbose=3)

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    lno_emp[ifrag] = eorb_mp
    lno_ecc[ifrag] = eorb_cc
    
    # lno_emp2[ifrag] = eorb_mp2


 ******* LNO-CALCULATION ******* 

LO occ span the occupied MO occ space - True.
MO occ span the occupied LO occ space - False.


====================== unrestricted LNO-FRAGMENT 1/(4,4) =======================
Center Atom O
PySCF NumPy Threads = 16
LAS occupied orbitals:  [7, 5]
LAS virtual orbitals:   [1, 3]
LAS total size:         [8, 8]

WARN: CCSD detected DF being used in the HF object. MO integrals are computed based on the DF 3-index tensors.
It's recommended to use dfccsd.CCSD for the DF-CCSD calculations


******** <class 'pyscf.lno.ulnoccsd.MODIFIED_UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(7), np.int64(5)), nmo = (8, 8)
frozen orbitals [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 19]), array([ 0,  1,  2,  3,  4,  5,  6,  7,  8, 17, 18, 19])]
max_cycle = 50
direct = 0
conv_tol = 1e-06
conv_tol_normt = 1e-05
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 20000 MB (current use 930 MB)
Init t2, MP2 energy = -0.0866168168954746
Init E_

In [11]:
print(lno_emp)
print(lno_ecc)

[-0.04857551 -0.04853125 -0.04853125 -0.04857551]
[-0.05446248 -0.05441234 -0.05441234 -0.05446248]


In [32]:
def make_ulas2(mlno, eris, orbloc, lno_type, lno_thresh=None):
    """
    Create localized active space for a given set of localized orbitals
    given in orbloc
    """
    log = logger.new_logger(mlno)

    if lno_thresh is None:
        thresh_occ, thresh_vir = mlno.lno_thresh
    else:
        thresh_occ, thresh_vir = lno_thresh

    s1e = mlno.s1e

    orboccfrz_core = [None,] * 2
    orbocc = [None,] * 2
    orbvir = [None,] * 2
    orbvirfrz_core = [None,] * 2
    moeocc = [None,] * 2
    moevir = [None,] * 2

    uocc_loc = [None,] * 2
    uocc_std = [None,] * 2
    uocc_orth = [None,] * 2

    mo_splits = mlno.split_mo_coeff()
    moe_splits = mlno.split_mo_energy()
    for s in range(2):
        orboccfrz_core[s], orbocc[s], orbvir[s], orbvirfrz_core[s] = mo_splits[s]
        moeocc[s], moevir[s] = moe_splits[s][1:3]
        
        #####################################
        # Projection of LO onto occ and vir #
        #####################################
        ovlp = reduce(np.dot, (orbloc[s].T.conj(), s1e, orbocc[s]))
        uocc_loc[s], uocc_std[s], uocc_orth[s] = \
            lno.projection_construction(ovlp, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
        # NOTE we allow empty fragments
        # if uocc_loc[s].shape[1] == 0:
        #    log.error('LOs do not overlap with occupied space. This could be caused '
        #              'by either a bad fragment choice or too high of `lo_proj_thresh_active` '
        #              '(current value: %s).', mlno.lo_proj_thresh_active)
        #    raise RuntimeError
        log.info('LO occ proj: %d active | %d standby | %d orthogonal',
                 *[u.shape[1] for u in [uocc_loc[s], uocc_std[s], uocc_orth[s]]])

    ####################
    # LNO construction #
    ####################
    if lno_type[0] == lno_type[1] == '1h':
        # NOTE: uvir_loc is not used in 1h/1h, so we pass None
        if getattr(mlno, 'with_df', None):
            dmoo, dmvv = ulno.make_lo_rdm1_1h_df(eris, moeocc, moevir, uocc_loc)
        else:
            dmoo, dmvv = ulno.make_lo_rdm1_1h(eris, moeocc, moevir, uocc_loc)
    else:
        raise NotImplementedError('Unsupported LNO type')
        
    # if mlno._match_oldulno:
    #     dmoo[0],dmoo[1]=dmoo[0]/2.0,dmoo[1]/2.0
    #     dmvv[0],dmvv[1]=dmvv[0]/2.0,dmvv[1]/2.0

    orbfrag = [None,] * 2
    frzfrag = [None,] * 2
    uoccact_loc = [None,] * 2
    can_orbfrag = [None,] * 2
    can_uoccact_loc = [None,] * 2
    frag_msg = ""

    for s in [0]:
        dmoo_orth = reduce(np.dot, (uocc_orth[s].T.conj(), dmoo[s], uocc_orth[s]))

        _param = lno_param[s][0]
        if _param['norb'] is not None:
            _param['norb'] -= uocc_loc[s].shape[1] + uocc_std[s].shape[1]

        uoccact_orth, uoccfrz_orth = lno.natorb_select(dmoo_orth, uocc_orth[s], thresh=thresh_occ)
        # uoccact_orth = uoccact_orth[:,::-1] # for occ. flip the NOs so they are in the
        # uoccfrz_orth = uoccfrz_orth[:,::-1] # order of small eigenvalue -> large eigenvalue
        # orboccfrz = np.hstack((orboccfrz_core[s], np.dot(orbocc[s], uoccfrz_orth)))
        # uoccact = np.hstack((uoccact_orth, uocc_std[s], uocc_loc[s]))
        # orboccact = np.dot(orbocc[s], uoccact)
        # uoccact_loc[s] = np.linalg.multi_dot((orboccact.T.conj(), s1e, orbloc[s]))
        # canonized
        # can_uoccact = lno.subspace_eigh(np.diag(moeocc[s]), np.hstack((uoccact_orth, uocc_std[s], uocc_loc[s])))[1]
        # can_orboccact = np.dot(orbocc[s], can_uoccact)
        # can_uoccact_loc[s] = np.linalg.multi_dot((can_orboccact.T.conj(), s1e, orbloc[s]))

    #     orbviract, orbvirfrz = lno.natorb_select(dmvv[s], orbvir[s], thresh=thresh_vir)
    #     orbvirfrz = np.hstack((orbvirfrz, orbvirfrz_core[s]))
    #     uviract = reduce(np.dot, (orbvir[s].T.conj(), s1e, orbviract))
    #     uviract = uviract
    #     orbviract = np.dot(orbvir[s], uviract)
    #     # canonized
    #     can_uviract = lno.subspace_eigh(np.diag(moevir[s]), uviract)[1]
    #     can_orbviract = np.dot(orbvir[s], can_uviract)

    #     ####################
    #     # LAS construction #
    #     ####################
    #     orbfragall = [orboccfrz, orboccact, orbviract, orbvirfrz]
    #     can_orbfragall = [orboccfrz, can_orboccact, can_orbviract, orbvirfrz]
    #     orbfrag[s] = orbfragall
    #     can_orbfrag[s] = np.hstack(can_orbfragall)
    #     norbfragall = np.asarray([x.shape[1] for x in orbfragall])
    #     locfragall = np.cumsum([0] + norbfragall.tolist()).astype(int)
    #     frzfrag[s] = np.concatenate((
    #         np.arange(locfragall[0], locfragall[1]),
    #         np.arange(locfragall[3], locfragall[4]))).astype(int)
    #     frag_msg += '\nSpin channel %d: %d/%d Occ | %d/%d Vir | %d/%d MOs\n' % (
    #                     s,
    #                     norbfragall[1], sum(norbfragall[:2]),
    #                     norbfragall[2], sum(norbfragall[2:4]),
    #                     sum(norbfragall[1:3]), sum(norbfragall)
    #                 )
    #     if len(frzfrag[s]) == 0:
    #         frzfrag[s] = 0

    # print(frag_msg)

    return uoccact_orth, uoccfrz_orth, dmoo_orth, uocc_orth[0]

In [36]:
lno_thresh = [1e-3, 1e-4]
uoccact_orth1, uoccfrz_orth1, dmoo_orth1, uocc_orth1 = make_ulas2(mlno, eris, orbloc, lno_type, lno_thresh)
print(uoccact_orth1.shape, uoccfrz_orth1.shape)
# print(orbfrag1[0][0].shape, orbfrag1[0][1].shape, orbfrag1[0][2].shape, orbfrag1[0][3].shape)
# print(orbfrag1[1][0].shape, orbfrag1[1][1].shape, orbfrag1[1][2].shape, orbfrag1[1][3].shape)

(14, 0) (14, 9)


In [37]:
lno_thresh = [1e-4, 1e-5]
uoccact_orth2, uoccfrz_orth2, dmoo_orth2, uocc_orth2 = make_ulas2(mlno, eris, orbloc, lno_type, lno_thresh)
print(uoccact_orth2.shape, uoccfrz_orth2.shape)
# print(orbfrag2[0][0].shape, orbfrag2[0][1].shape, orbfrag2[0][2].shape, orbfrag2[0][3].shape)
# print(orbfrag2[1][0].shape, orbfrag2[1][1].shape, orbfrag2[1][2].shape, orbfrag2[1][3].shape)

(14, 2) (14, 7)


In [38]:
print(abs(dmoo_orth1 - dmoo_orth2).max())
print(abs(uocc_orth1 - uocc_orth2).max())

9.41900637346782e-19
0.0


In [81]:
def natorb_select(dm, orb, thresh, pct_occ=None, norb=None):
    e, u = np.linalg.eigh(dm)
    e = abs(e)
    order = np.argsort(e)[::-1]
    e = e[order]
    u = u[:,order]
    if norb is None:
        if pct_occ is None:
            nkeep = np.count_nonzero(e > thresh)
        else:
            nkeep = np.count_nonzero(np.cumsum(e)/np.sum(e) <= pct_occ)
    else:
        nkeep = min(max(norb, 0), e.size)

    idx  = np.arange(0,     nkeep,  dtype=int)
    idxc = np.arange(nkeep, e.size, dtype=int)
    orbx = np.dot(orb, u)
    orb1x = sub_colspace(orbx, idx)
    orb0x = sub_colspace(orbx, idxc)
    return orb1x, orb0x, e

def sub_colspace(A, idx):
    if idx.size == 0:
        return np.zeros([A.shape[0],0])
    else:
        return A[:,idx]

In [85]:
uoccact_orth3, uoccfrz_orth3, e3 = natorb_select(dmoo_orth1, uocc_orth1, thresh=1e-3)
print(uoccact_orth3.shape, uoccfrz_orth3.shape)

(14, 0) (14, 9)


In [84]:
uoccact_orth4, uoccfrz_orth4, e4 = natorb_select(dmoo_orth2, uocc_orth2, thresh=1e-3)
print(uoccact_orth4.shape, uoccfrz_orth4.shape)

(14, 0) (14, 9)


In [86]:
print(e3)

[3.21763480e-04 3.21763480e-04 1.43837164e-05 1.68833356e-07
 8.27977821e-10 8.27977821e-10 3.15751581e-10 3.66681682e-11
 3.66681682e-11]


In [77]:
print(uoccfrz_orth3[:2,::-1])

[[-4.86783223e-11 -7.49948343e-11 -3.74459789e-01 -3.14778905e-12
  -1.64855418e-12  5.95489751e-01 -9.69947535e-02  1.95425815e-14
   1.14127898e-14]
 [ 4.86757781e-11  7.49555812e-11  3.74276344e-01  3.14400632e-12
   1.64191699e-12 -5.91785893e-01  7.50865338e-02  1.41028682e-14
   7.97388685e-15]]


In [78]:
n = 1
print(abs(uoccfrz_orth3[:,::-1][:,:n] - uoccfrz_orth4[:,::-1][:,:n]).max())

1.3458214751350899


In [79]:
print(uoccfrz_orth3[:,::-1][:,0])

[-4.86783223e-11  4.86757781e-11  4.51077353e-11  4.51109809e-11
 -6.20527196e-01 -3.38784674e-01  6.43814210e-01 -2.92092801e-01
 -4.29116465e-12 -4.26703464e-12  1.21515182e-02  5.46087689e-03
 -1.21765686e-02  5.50019469e-03]


In [80]:
print(uoccfrz_orth4[:,::-1][:,0])

[ 5.09073468e-12 -5.10735421e-12  2.29270353e-11  2.29113412e-11
  7.06276261e-01 -3.16685945e-02 -7.02007265e-01 -8.36704380e-02
 -4.04396308e-12 -4.04709252e-12 -1.32228492e-02  1.62383282e-03
  1.32646441e-02  1.60317871e-03]
